In [20]:
import torch, pickle
from pathlib import Path
from nanochat.gpt import GPT, GPTConfig
import chess, chess.pgn, chess.engine

In [21]:
ckpt = torch.load("models/chess_min_gpu_large.pt", map_location="cpu", weights_only=False)
config = GPTConfig(**ckpt["meta"]["model_config"])
model = GPT(config).eval()
model.load_state_dict(ckpt["model"])
stoi = ckpt["meta"]["tokenizer"]["stoi"]
itos = ckpt["meta"]["tokenizer"]["itos"]
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

In [22]:
engine = chess.engine.SimpleEngine.popen_uci("fairy-stockfish")
engine.configure({ "UCI_LimitStrength": True, "UCI_Elo": 500})

In [23]:
board = chess.Board()
result = engine.play(board, chess.engine.Limit(time=0.1))
board.san(result.move)



'd3'

In [24]:
def play_game(model_color):
    board = chess.Board()
    tokens = ["<bos>"]

    while not board.is_game_over():
        if board.turn == model_color:
            # Filter tokens to only include those in vocabulary - figure out how to handle this gracefully
            token_ids = [stoi[t] for t in tokens if t in stoi]
            if not token_ids:
                token_ids = [stoi["<bos>"]]
            x = torch.tensor(token_ids, device=device)[None, :]
            logits = model(x[:, -config.sequence_len:])

            legal_san = {board.san(mv) for mv in board.legal_moves}
            mask = torch.full((len(itos),), float("-inf"), device=device)
            for token, idx in stoi.items():
                if token in legal_san:
                    mask[idx] = logits[0, -1, idx]

            probs = torch.softmax(mask, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            move_san = itos[next_id.item()]

        else:
            result = engine.play(board, chess.engine.Limit(time=0.1))
            move_san = board.san(result.move)

        board.push_san(move_san)
        tokens.append(move_san)

    return board.outcome(), tokens

In [25]:
def evaluate(num_games=100):
    results = {"win": 0, "loss": 0, "draw": 0}

    for i in range(num_games):
        model_color = chess.WHITE if i % 2 == 0 else chess.BLACK
        outcome, game_tokens = play_game(model_color)
       # print(f"Model color: {'WHITE' if model_color else 'BLACK'}")
        #print(f"{' '.join(game_tokens[1:])}")

        if outcome.winner is None:
            results["draw"] += 1
        elif outcome.winner == model_color:
            results["win"] += 1
        else:
            results["loss"] += 1

        if (i + 1) % 10 == 0:
            print(f"Game {i+1}/{num_games}: {results}")

    total = sum(results.values())
    win_rate = (results["win"] + 0.5 * results["draw"]) / total
    print(f"\nFinal: {results}")
    print(f"Win rate: {win_rate:.1%}")

    return results

In [26]:
evaluate(num_games=100)

Game 10/100: {'win': 6, 'loss': 0, 'draw': 4}
Game 20/100: {'win': 14, 'loss': 0, 'draw': 6}
Game 30/100: {'win': 19, 'loss': 0, 'draw': 11}
Game 40/100: {'win': 27, 'loss': 0, 'draw': 13}
Game 50/100: {'win': 32, 'loss': 0, 'draw': 18}
Game 60/100: {'win': 40, 'loss': 0, 'draw': 20}
Game 70/100: {'win': 47, 'loss': 0, 'draw': 23}
Game 80/100: {'win': 51, 'loss': 3, 'draw': 26}
Game 90/100: {'win': 60, 'loss': 3, 'draw': 27}
Game 100/100: {'win': 67, 'loss': 3, 'draw': 30}

Final: {'win': 67, 'loss': 3, 'draw': 30}
Win rate: 82.0%


{'win': 67, 'loss': 3, 'draw': 30}